# Estimator-input audit -- what the VIO actually received, per flight

Everything here is the **input** side: what the tracker emitted, on what clock, and whether
the local geometry in it is any good. The estimator is out of the loop -- for the pose
comparison see [`vio-quality.ipynb`](vio-quality.ipynb), and for OAK-D-IMU-vs-FC-IMU
alignment see [`vio-input-alignment.ipynb`](vio-input-alignment.ipynb).

- **Input:** the FC `.bin` (parameter `input_file`); the `.feat` fixture and capture session
  are resolved as siblings under `captures/`.
- **Modules:** `capture_align`, `feat_stream`, `local_pose`, `still_banding` -- all in
  `analysis/`, all covered by `analysis/test_analysis_modules.py`.
- **Evidence rows this regenerates:** E30 (clock tie), E31 (frame loss), E32 (feature supply
  and depth), E34 (still banding), E35 (local pose) in
  [`vio-quality-experiments.md`](vio-quality-experiments.md). Frame loss is also
  coordinator [#156](https://github.com/symmatree/coordinator/issues/156).


In [ ]:
# parameters -- papermill overrides these per flight
input_file = "/home/jovyan/datasets/flights/rekon10/260814-woods/1980-01-11 08-00-07.bin"
debug = False
run_banding = True      # the still-banding pass is the slow one (image IO + per-pair fits)
banding_max_lag = 2     # pair each still with the next N stills


In [ ]:
import json, sys, hashlib, datetime, glob, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# The analysis modules sit next to this notebook. Cover running from analysis/, the repo
# root, or a papermill workdir one level down (same bootstrap as vio-quality.ipynb).
_here = Path.cwd()
for _cand in (_here, _here / "analysis", _here.parent):
    if (_cand / "feat_stream.py").exists():
        sys.path.insert(0, str(_cand)); break

import capture_align as ca
import feat_stream as fs
import local_pose as lp
import still_banding as sb
from ardupilot_log import parse_log

plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

LOG = Path(input_file)
D = LOG.parent
feats_found = sorted(D.glob("captures/*/*/*.feat"))
sessions = sorted({Path(p).parent for p in feats_found})
print("flight   :", D.name)
print("FC log   :", LOG.name, f"({LOG.stat().st_size/1e6:.0f} MB)")
print("sessions :", [s.name for s in sessions] or "(none)")


In [ ]:
# --- ASSUMPTIONS: refuse to produce numbers from an input we cannot vouch for ---
# A violated assumption halts the notebook. "Unassessable" is the honest output; a green
# verdict computed from a stream we cannot characterise is the failure mode to avoid.
issues = []

assert feats_found, f"no .feat fixture under {D}/captures -- nothing to audit"
FEAT = feats_found[0]
SESSION = FEAT.parent
if len(feats_found) > 1:
    issues.append(f"{len(feats_found)} .feat fixtures present; auditing {FEAT.name} only")

F, fc_duration, parms = parse_log(LOG, ["VISP", "VISV", "XKF1", "CTUN", "ARM", "ATT", "ESC", "GPS"])

# The clock tie is derived from VISP.RTimeUS (E30). Without VISP there is no way to put the
# capture on the FC clock from this log alone, and every windowed number below is unfounded.
assert "VISP" in F and len(F["VISP"]) > 100, "no usable VISP in the FC log -- cannot tie capture to FC time"

imu, features = fs.decode(FEAT)
assert len(features) > 100, f"only {len(features)} feature datagrams in {FEAT.name}"

# frame_loss only means something if the device timestamps really are on a frame grid.
loss = fs.frame_loss(features)
assert loss["grid_residual_s"] < 1e-4, (
    f"device timestamps are not on a frame grid (residual {loss['grid_residual_s']:.2e} s) "
    "-- the frame-loss count would be meaningless")

window = ca.airborne_window(F)
assert window is not None, "could not derive an airborne window (no altitude excursion in XKF1/CTUN)"
LIFT, LAND = window

print("=== ASSUMPTIONS ===")
print(f"  fixture      : {FEAT.name}  ({len(features)} feature datagrams, {len(imu)} IMU)")
print(f"  frame grid   : {loss['frame_step_s']*1000:.1f} ms step, residual {loss['grid_residual_s']:.2e} s")
print(f"  airborne     : {LIFT:.1f} - {LAND:.1f} s  ({LAND-LIFT:.1f} s of a {fc_duration:.0f} s log)")
for i in issues:
    print("  WARNING:", i)
if not issues:
    print("  all ok")


## Timing: putting the capture on the FC clock (E30)

The coordinator wall clock steps once, when NTP first disciplines it after boot. Join on
**monotonic** and repair the pre-step wall stamps from it; the bridge to FC time comes from
`VISP.RTimeUS` (the coordinator's own stamp, logged by the FC next to its own `TimeUS`), so
no GPS week/ms arithmetic is involved.

In [ ]:
import struct
FRAME = struct.Struct("<ddHI")
_raw = open(FEAT, "rb").read()
_mono, _wall, _off = [], [], 0
while _off + FRAME.size <= len(_raw):
    _tm, _tu, _sid, _ln = FRAME.unpack_from(_raw, _off); _off += FRAME.size
    if _off + _ln > len(_raw): break
    if _sid == fs.SID_FEATURES: _mono.append(_tm); _wall.append(_tu)
    _off += _ln
_mono, _wall = np.array(_mono), np.array(_wall)

step = ca.clock_step(_mono, _wall)
to_fc, fit = ca.fc_time_from_visp(F["VISP"])
t_fc = to_fc(ca.repair_wall_clock(_mono, _wall))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
ax[0].plot(_mono, _wall - _mono - (_wall - _mono)[-1], ".", ms=2)
ax[0].set_xlabel("coordinator monotonic (s)"); ax[0].set_ylabel("wall - monotonic\n(offset from final, s)")
ax[0].set_title("NTP step" if step else "no clock step"); ax[0].grid(alpha=.3)
if step:
    ax[0].axvline(step["at_monotonic_s"], color="crimson", lw=1)
    ax[0].text(0.02, 0.5, f"+{step['step_s']:.3f} s\nat mono {step['at_monotonic_s']:.1f}",
               transform=ax[0].transAxes, fontsize=8, color="crimson")
_r = np.asarray(F["VISP"]["t_s"]) - to_fc(np.asarray(F["VISP"]["RTimeUS"]) / 1e6)
ax[1].plot(np.asarray(F["VISP"]["t_s"]), _r * 1000, ".", ms=1)
ax[1].set_xlabel("FC t_s (s)"); ax[1].set_ylabel("fit residual (ms)")
ax[1].set_title(f"coordinator wall -> FC time  (rms {fit['residual_rms_s']*1000:.2f} ms)")
ax[1].grid(alpha=.3); plt.tight_layout(); plt.show()

print("clock step  :", {k: round(v, 4) if isinstance(v, float) else v for k, v in (step or {}).items()})
print("fc time fit :", {k: (round(v, 6) if isinstance(v, float) else v) for k, v in fit.items()})
print(f"capture spans FC t {t_fc.min():.1f} .. {t_fc.max():.1f} s")


## What the estimator actually received (E31, [#156](https://github.com/symmatree/coordinator/issues/156))

Feature-frame device timestamps sit on the sensor's frame grid, so frames skipped between two
emitted datagrams are countable. This is the tracker's own output, upstream of the estimator
and of anything the FC logs.

In [ ]:
air = (t_fc >= LIFT) & (t_fc <= LAND)
feat_air = [f for f, k in zip(features, air) if k]
t_air = t_fc[air]
loss_air = fs.frame_loss(feat_air)

gaps = np.diff(t_air)
duty = float(gaps[gaps <= 0.3].sum() / (t_air[-1] - t_air[0]))

fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(t_air[:-1], gaps, ".", ms=2, color="purple")
for thr, c in [(0.3, "orange"), (1.0, "red")]:
    ax[0].axhline(thr, color=c, ls="--", lw=.8)
ax[0].set_yscale("log"); ax[0].set_ylabel("gap to next\nfeature frame (s)"); ax[0].grid(alpha=.3)
ax[0].set_title(f"{loss_air['loss_frac']*100:.0f}% of frames never emitted"
                f"  |  stream up {duty*100:.0f}% of the airborne window")
xk = F["XKF1"]; xk0 = xk[xk["C"] == 0].sort_values("t_s")
ax[1].plot(xk0["t_s"], -xk0["PD"], "r", lw=1, label="EKF alt")
for g0, g in zip(t_air[:-1], gaps):
    if g > 1.0: ax[1].axvspan(g0, g0 + g, color="purple", alpha=.25)
ax[1].set_xlim(LIFT, LAND); ax[1].set_ylabel("alt (m)"); ax[1].set_xlabel("FC t_s (s)")
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
ax[1].set_title("shaded = gaps > 1 s", fontsize=9)
plt.tight_layout(); plt.show()

print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in loss_air.items()}, indent=2))


## Feature supply and scene depth (E32, T12)

Counts are what T6 is about; the disparity/depth distribution is what T12 is about. Both are
airborne-gated -- on-ground frames look at the ground a metre away and shift both.

In [ ]:
counts = fs.feature_counts(feat_air)
depth = fs.depth_stats(feat_air)
disp_all, z_all = [], []
for f in feat_air:
    if len(f[2]) < 3: continue
    k = fs.intrinsics(f[2])
    _, xyz, d = fs.triangulate(f[2])
    disp_all.append(d * (k[0] if k else np.nan)); z_all.append(xyz[:, 2])
disp_all = np.concatenate(disp_all); z_all = np.concatenate(z_all)

fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
ax[0].plot(t_air, counts, ".", ms=2, alpha=.4)
_w = max(5, len(counts)//50)
ax[0].plot(t_air, np.convolve(counts, np.ones(_w)/_w, "same"), "b", lw=1.5)
ax[0].axhline(80, color="green", ls="--", lw=.8, label="setNumTargetFeatures")
ax[0].axhline(10, color="red", ls=":", lw=1)
ax[0].set_xlabel("FC t_s (s)"); ax[0].set_ylabel("stereo-matched features/frame")
ax[0].set_ylim(0, 90); ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)
ax[1].hist(disp_all, bins=np.linspace(0, 20, 60), color="steelblue")
ax[1].axvline(2, color="red", ls="--", lw=1)
ax[1].set_xlabel("disparity (px)"); ax[1].set_ylabel("features")
ax[1].set_title(f"{depth['frac_under_2px']*100:.0f}% under 2 px", fontsize=9); ax[1].grid(alpha=.3)
ax[2].hist(np.clip(z_all, 0, 80), bins=60, color="darkorange")
ax[2].axvline(10, color="red", ls="--", lw=1)
ax[2].set_xlabel("triangulated depth (m)")
ax[2].set_title(f"median {depth['depth_m']['median']:.1f} m, "
                f"{depth['frac_beyond_10m']*100:.0f}% beyond 10 m", fontsize=9); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"features/frame: median {np.median(counts):.0f}  p5 {np.percentile(counts,5):.0f}  min {counts.min()}")
print(json.dumps(depth, indent=2))


## Local relative pose -- VINS out of the loop (E35)

Frame-to-frame rigid pose from the tracker's own features: rotation from bearings (no depth),
translation from a depth-scaled RANSAC. The floor test -- if these deltas are bad, nothing
downstream rescues them. Integration breaks at every dropped-frame gap, so the output is
segments, never one track.

In [ ]:
poses = lp.relative_poses(feat_air, t_air)
solved = [p for p in poses if p["ok"]]
segments = lp.integrate(poses)
print(f"pairs {len(poses)}, solved {len(solved)} ({100*len(solved)/max(1,len(poses)):.0f}%)  "
      f"-> {len(segments)} segments, longest {max((s['t'][-1]-s['t'][0]) for s in segments):.1f} s")

def _umeyama_ate(P, G):
    R, c, t = lp.umeyama(P, G)
    return float(np.sqrt((np.linalg.norm((c * (P @ R.T) + t) - G, axis=1) ** 2).mean())), float(c)

vp = F["VISP"].sort_values("t_s")
rows = []
for s in segments:
    tt, P = s["t"], s["positions"]
    G = np.column_stack([np.interp(tt, xk0["t_s"], xk0["PN"]), np.interp(tt, xk0["t_s"], xk0["PE"]),
                         np.interp(tt, xk0["t_s"], -xk0["PD"])])
    extent = float(np.linalg.norm(G - G.mean(0), axis=1).max() * 2)
    ate, scale = _umeyama_ate(P, G)
    V = vp[(vp["t_s"] >= tt[0]) & (vp["t_s"] <= tt[-1])]
    if len(V) > 10:
        Vp = np.column_stack([V["PX"], V["PY"], -V["PZ"]]); tv = V["t_s"].values
        Gv = np.column_stack([np.interp(tv, xk0["t_s"], xk0["PN"]), np.interp(tv, xk0["t_s"], xk0["PE"]),
                              np.interp(tv, xk0["t_s"], -xk0["PD"])])
        v_ate, v_scale = _umeyama_ate(Vp, Gv)
    else:
        v_ate = v_scale = np.nan
    rows.append(dict(t0=float(tt[0]), dur=float(tt[-1]-tt[0]), extent_m=extent,
                     local_ate_m=ate, local_scale=scale, visp_ate_m=v_ate, visp_scale=v_scale))

fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
ax[0].plot(t_air[1:][[p["ok"] for p in poses]], [p["inliers"] for p in solved], ".", ms=2)
ax[0].set_xlabel("FC t_s (s)"); ax[0].set_ylabel("RANSAC inliers"); ax[0].grid(alpha=.3)
ax[0].set_title("solve quality", fontsize=9)
ax[1].plot([p["depth_median_m"] for p in solved], [p["residual_m"] for p in solved], ".", ms=2, alpha=.4)
ax[1].set_xlabel("scene depth (m)"); ax[1].set_ylabel("3D-3D residual (m)")
ax[1].set_xscale("log"); ax[1].set_yscale("log"); ax[1].grid(alpha=.3)
ax[1].set_title("residual grows with depth (T12)", fontsize=9)
big = [r for r in rows if r["extent_m"] > 2]
if big:
    ax[2].scatter([r["extent_m"] for r in big], [r["local_ate_m"] for r in big], label="local", s=25)
    ax[2].scatter([r["extent_m"] for r in big], [r["visp_ate_m"] for r in big], label="VISP", s=25, marker="x")
    ax[2].set_xlabel("segment true extent (m)"); ax[2].set_ylabel("aligned ATE (m)")
    ax[2].legend(fontsize=8); ax[2].grid(alpha=.3)
    ax[2].set_title("per segment, >2 m extent only", fontsize=9)
plt.tight_layout(); plt.show()

# Segments below ~2 m of true extent are Umeyama-degenerate (E26); they are excluded from the
# medians rather than quietly averaged in.
local_summary = {}
if big:
    local_summary = {k: round(float(np.median([r[k] for r in big])), 3)
                     for k in ("local_ate_m", "visp_ate_m", "local_scale", "visp_scale", "extent_m", "dur")}
    local_summary["n_segments_over_2m"] = len(big)
    print(json.dumps(local_summary, indent=2))
else:
    print("no segment exceeds 2 m of true extent -- every Umeyama fit here would be degenerate")


## Still banding (E34)

Scene-cancelled: register two stills of the same scene, ratio their per-row gradient energy,
fit a pitch. The pitch is a **spatial** period in rows; converting it to Hz needs the sensor
readout time, which we have not measured, so it is not done here.

In [ ]:
stills = sorted(glob.glob(str(SESSION / "*.jpg")))
banding = {"n_stills": len(stills), "skipped": not (run_banding and stills)}
if run_banding and stills:
    banding.update(sb.session_pitch(stills, max_lag=banding_max_lag))
    pairs = banding.pop("pairs", [])
    if pairs:
        best = max(pairs, key=lambda p: p["r2"])
        a, b = sb.load_gray(stills[best["i"]]), sb.load_gray(stills[best["j"]])
        ps, ve, _ = sb.band_pitch(a, b)
        from scipy.ndimage import uniform_filter1d
        y0, y1, x0, x1 = 80, 2960, 700, 3300
        dy, dx = best["dy"], best["dx"]
        r = np.log(sb.row_energy(b[y0+dy:y1+dy, x0+dx:x1+dx]) / sb.row_energy(a[y0:y1, x0:x1]))
        r = r - uniform_filter1d(r, int(2500/sb.ROW_BLOCK) | 1)
        rr = np.arange(len(r)) * sb.ROW_BLOCK + y0
        fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
        ax[0].plot(rr, np.exp(r), "k", lw=.8); ax[0].axhline(1, color="grey", ls="--")
        ax[0].set_yscale("log"); ax[0].set_xlabel("image row")
        ax[0].set_ylabel("per-row sharpness ratio\n(scene cancels)"); ax[0].grid(alpha=.3)
        ax[0].set_title(f"stills {best['i']} vs {best['j']}", fontsize=9)
        ax[1].plot(ps, ve, "b", lw=1); ax[1].axvline(best["pitch_rows"], color="crimson", ls="--")
        ax[1].set_xlabel("trial pitch (rows)"); ax[1].set_ylabel("variance explained")
        ax[1].grid(alpha=.3)
        ax[1].set_title(f"peak {best['pitch_rows']:.0f} rows, R^2 {best['r2']:.2f}"
                        "  (inspect for a peak at the bound)", fontsize=9)
        plt.tight_layout(); plt.show()
print(json.dumps({k: v for k, v in banding.items() if k != "pairs"}, indent=2))


## Contact sheet

Stills in flight order, so the scene and exposure story is visible next to the numbers above.

In [ ]:
if stills:
    from PIL import Image
    n = min(len(stills), 24)
    pick = [stills[i] for i in np.linspace(0, len(stills)-1, n).astype(int)]
    cols = 6; rows_n = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(2.6*cols, 2.0*rows_n))
    for k, axx in enumerate(np.atleast_1d(axes).ravel()):
        axx.axis("off")
        if k >= n: continue
        im = Image.open(pick[k]); im.draft("RGB", (400, 300))
        axx.imshow(np.asarray(im.convert("RGB")))
        axx.set_title(Path(pick[k]).name.split("_")[1], fontsize=6)
    plt.tight_layout(); plt.show()
else:
    print("no stills in this capture session")


## Provenance sidecar

Emitted alongside the input (`alongside-input` placement), so the numbers above are readable
without a kernel and citable from the experiments doc.

In [ ]:
out = {
    "notebook": "estimator-input-audit.ipynb",
    "run_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "debug": debug,
    "inputs": {
        "fc_bin": str(LOG), "fc_bin_sha256": hashlib.sha256(LOG.read_bytes()).hexdigest(),
        "feat": str(FEAT), "feat_sha256": hashlib.sha256(FEAT.read_bytes()).hexdigest(),
        "session": str(SESSION),
    },
    "airborne_window_s": [round(LIFT, 2), round(LAND, 2)],
    "clock_step": step,
    "fc_time_fit": fit,
    "frame_loss_airborne": loss_air,
    "features_per_frame": {"median": float(np.median(counts)), "p5": float(np.percentile(counts, 5)),
                           "min": int(counts.min()), "max": int(counts.max())},
    "depth": depth,
    "local_pose": {"pairs": len(poses), "solved": len(solved), "segments": len(segments), **local_summary},
    "banding": banding,
    "assumption_warnings": issues,
    "status": "warnings" if issues else "ok",
}
dest = D / "estimator-input-audit.json"
dest.write_text(json.dumps(out, indent=2, default=float))
print("wrote", dest)
print(json.dumps(out, indent=2, default=float)[:1500])
